In [2]:
from google.colab import userdata, files
import os, subprocess, json, glob, zipfile
import pandas as pd

GH_USER, REPO = "preranaprakash129-netizen", "phishing-url-gnn-detection"

# 1) clone the repo (token is removed from the saved remote URL)
if not os.path.exists(REPO):
    token = userdata.get("GH_token")
    r = subprocess.run(["git", "clone", f"https://{token}@github.com/{GH_USER}/{REPO}.git"],
                       capture_output=True, text=True)
    print("clone:", "ok" if r.returncode == 0 else "FAILED -> check Notebook access toggle / token expiry")
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://github.com/{GH_USER}/{REPO}.git"])

# 2) legacy training data (same construction as before, with the corrected Kaggle labels)
k = pd.read_csv(f"{REPO}/kaggle_cleaned.csv", usecols=["URL", "label"]).rename(columns={"URL": "url"})
k["label"] = k["label"].map({1: "legitimate", 0: "phishing"})
g1 = pd.read_csv(f"{REPO}/cleaned_github_dataset1.csv")[["url", "label"]]
g2 = pd.DataFrame(json.load(open(f"{REPO}/cleaned_github_dataset_2.json")))[["url", "label"]]
legacy = pd.concat([k, g1, g2], ignore_index=True)
legacy["url"] = legacy["url"].astype(str).str.strip()
legacy = legacy.drop_duplicates(subset="url").reset_index(drop=True)
print("legacy:", legacy.shape, legacy["label"].value_counts().to_dict())

# 3) fresh crawl: from the repo zip, or upload it if it isn't there
os.makedirs("fresh", exist_ok=True)
zips = glob.glob(f"{REPO}/fresh_backup*.zip")
if zips:
    zf = max(zips, key=os.path.getsize)          # largest = the backup taken after both crawls
else:
    print("no zip in the repo -> choose fresh_backup.zip from your Downloads folder")
    zf = list(files.upload())[0]
zipfile.ZipFile(zf).extractall("fresh")

recs = [json.loads(l) for p in sorted(glob.glob("fresh/crawl_*.jsonl")) for l in open(p)]
fresh_phish = sorted({r["original_url"] for r in recs if r["klass"] == "phish"})
fresh_home  = sorted({r["original_url"] for r in recs if r["klass"] == "legit"})
fresh_paths = sorted({l for r in recs if r["klass"] == "legit" for l in r["internal_links"]})
print("records:", len(recs), "| phish:", len(fresh_phish), "| homepages:", len(fresh_home),
      "| legit with paths:", len(fresh_paths))

clone: ok
legacy: (743518, 2) {'legitimate': 478802, 'phishing': 264716}
records: 600 | phish: 300 | homepages: 300 | legit with paths: 842


In [3]:
!pip -q install tldextract

import re, random, numpy as np, pandas as pd
from urllib.parse import urlparse
import tldextract
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_curve, roc_auc_score

# ---- clusters: campaign key for phishing, site for legit ----
ext = tldextract.TLDExtract(suffix_list_urls=())
def regdom(u):
    e = ext(u)
    return getattr(e, "top_domain_under_public_suffix", None) or e.registered_domain or (urlparse(u).hostname or "")
def campaign_key(u):
    path = re.sub(r"[0-9]+", "0", re.sub(r"[a-z0-9]{10,}", "X", urlparse(u).path.lower()))
    return regdom(u) + "|" + path

phish_cl = np.array([campaign_key(u) for u in fresh_phish])
home_cl  = np.array([regdom(u) for u in fresh_home])
paths_cl = np.array([regdom(u) for u in fresh_paths])
print("clusters -> phishing:", len(set(phish_cl)), "| homepage sites:", len(set(home_cl)),
      "| path-URL sites:", len(set(paths_cl)))

# ---- three training variants (same recipes as Steps 8 and 9) ----
hv = HashingVectorizer(analyzer="char", ngram_range=(1, 4), n_features=2**20, alternate_sign=False)
canon = lambda u: re.sub(r"^www\.", "", re.sub(r"^https?://", "", str(u).lower().strip())).rstrip("/")
host = lambda u: re.split(r"[/?#]", canon(u))[0]

def train_text(u, rng, mode):
    if mode == "raw": return u
    c = host(u) if mode == "hostonly" else canon(u)
    return rng.choice(["http://", "https://"]) + rng.choice(["", "www."]) + c

def test_text(u, mode):
    return "https://" + host(u) if mode == "hostonly" else u

tr = legacy.sample(200_000, random_state=0)
ytr = (tr["label"] == "phishing").astype(int)
scores = {}
for mode in ("raw", "augmented", "hostonly"):
    rng = random.Random(0)
    clf = SGDClassifier(loss="log_loss", alpha=1e-6, random_state=0).fit(
        hv.transform([train_text(u, rng, mode) for u in tr["url"]]), ytr)
    P = lambda urls: clf.predict_proba(hv.transform([test_text(u, mode) for u in urls]))[:, 1]
    scores[mode] = {"phish": P(fresh_phish), "home": P(fresh_home), "paths": P(fresh_paths)}
    print("trained", mode)

# ---- cluster bootstrap ----
def groups(cl):
    d = {}
    for i, c in enumerate(cl): d.setdefault(c, []).append(i)
    return [np.array(v) for v in d.values()]

def summarize(pos, neg, pcl, ncl, B=1000, seed=0):
    def metrics(p, n):
        y = np.r_[np.ones(len(p)), np.zeros(len(n))]; s = np.r_[p, n]
        fpr, tpr, _ = roc_curve(y, s)
        return {"TPR@5%FPR": np.interp(0.05, fpr, tpr), "TPR@10%FPR": np.interp(0.10, fpr, tpr),
                "AUC": roc_auc_score(y, s)}
    point = metrics(pos, neg)
    rng = np.random.default_rng(seed); PG, NG = groups(pcl), groups(ncl)
    runs = {k: [] for k in point}
    for _ in range(B):
        pi = np.concatenate([PG[j] for j in rng.integers(0, len(PG), len(PG))])
        ni = np.concatenate([NG[j] for j in rng.integers(0, len(NG), len(NG))])
        for k, v in metrics(pos[pi], neg[ni]).items(): runs[k].append(v)
    return {k: f"{point[k]:.2f} [{np.percentile(runs[k], 2.5):.2f}, {np.percentile(runs[k], 97.5):.2f}]"
            for k in point}

rows = {}
for mode in scores:
    for name, ncl in (("home", home_cl), ("paths", paths_cl)):
        rows[(mode, "vs legit " + name)] = summarize(scores[mode]["phish"], scores[mode][name], phish_cl, ncl)
print(pd.DataFrame(rows).T.to_string())

clusters -> phishing: 164 | homepage sites: 300 | path-URL sites: 173
trained raw
trained augmented
trained hostonly
                                  TPR@5%FPR         TPR@10%FPR                AUC
raw       vs legit home   0.00 [0.00, 0.01]  0.01 [0.00, 0.06]  0.52 [0.43, 0.61]
          vs legit paths  0.41 [0.20, 0.56]  0.58 [0.43, 0.70]  0.82 [0.75, 0.87]
augmented vs legit home   0.71 [0.56, 0.81]  0.77 [0.65, 0.87]  0.93 [0.89, 0.95]
          vs legit paths  0.45 [0.29, 0.58]  0.55 [0.42, 0.68]  0.85 [0.79, 0.89]
hostonly  vs legit home   0.58 [0.40, 0.69]  0.61 [0.45, 0.75]  0.86 [0.78, 0.92]
          vs legit paths  0.58 [0.41, 0.70]  0.61 [0.45, 0.75]  0.88 [0.80, 0.93]


In [4]:
from google.colab import files
res = pd.DataFrame(rows).T
res.to_csv("results_cluster_bootstrap.csv")
print(res.to_string())
files.download("results_cluster_bootstrap.csv")

                                  TPR@5%FPR         TPR@10%FPR                AUC
raw       vs legit home   0.00 [0.00, 0.01]  0.01 [0.00, 0.06]  0.52 [0.43, 0.61]
          vs legit paths  0.41 [0.20, 0.56]  0.58 [0.43, 0.70]  0.82 [0.75, 0.87]
augmented vs legit home   0.71 [0.56, 0.81]  0.77 [0.65, 0.87]  0.93 [0.89, 0.95]
          vs legit paths  0.45 [0.29, 0.58]  0.55 [0.42, 0.68]  0.85 [0.79, 0.89]
hostonly  vs legit home   0.58 [0.40, 0.69]  0.61 [0.45, 0.75]  0.86 [0.78, 0.92]
          vs legit paths  0.58 [0.41, 0.70]  0.61 [0.45, 0.75]  0.88 [0.80, 0.93]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from google.colab import userdata, files
import os, subprocess, json, glob, zipfile
import pandas as pd

GH_USER, REPO = "preranaprakash129-netizen", "phishing-url-gnn-detection"

if not os.path.exists(REPO):
    token = userdata.get("GH_token")
    r = subprocess.run(["git", "clone", f"https://{token}@github.com/{GH_USER}/{REPO}.git"],
                       capture_output=True, text=True)
    print("clone:", "ok" if r.returncode == 0 else "FAILED")
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://github.com/{GH_USER}/{REPO}.git"])

k = pd.read_csv(f"{REPO}/kaggle_cleaned.csv", usecols=["URL", "label"]).rename(columns={"URL": "url"})
k["label"] = k["label"].map({1: "legitimate", 0: "phishing"})
g1 = pd.read_csv(f"{REPO}/cleaned_github_dataset1.csv")[["url", "label"]]
g2 = pd.DataFrame(json.load(open(f"{REPO}/cleaned_github_dataset_2.json")))[["url", "label"]]
legacy = pd.concat([k, g1, g2], ignore_index=True)
legacy["url"] = legacy["url"].astype(str).str.strip()
legacy = legacy.drop_duplicates(subset="url").reset_index(drop=True)
print("legacy:", legacy.shape)

os.makedirs("fresh", exist_ok=True)
zf = max(glob.glob(f"{REPO}/fresh_backup*.zip"), key=os.path.getsize)
zipfile.ZipFile(zf).extractall("fresh")

recs = [json.loads(l) for p in sorted(glob.glob("fresh/crawl_*.jsonl")) for l in open(p)]
fresh_phish = sorted({r["original_url"] for r in recs if r["klass"] == "phish"})
fresh_home  = sorted({r["original_url"] for r in recs if r["klass"] == "legit"})
fresh_paths = sorted({l for r in recs if r["klass"] == "legit" for l in r["internal_links"]})
print("records:", len(recs), "| phish:", len(fresh_phish), "| homepages:", len(fresh_home),
      "| legit with paths:", len(fresh_paths))

clone: ok
legacy: (743518, 2)
records: 600 | phish: 300 | homepages: 300 | legit with paths: 842


In [2]:
!pip -q install tldextract

import re, random, numpy as np, pandas as pd
from urllib.parse import urlparse
import tldextract
from google.colab import files
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_curve, roc_auc_score

ext = tldextract.TLDExtract(suffix_list_urls=())
def regdom(u):
    e = ext(u)
    return getattr(e, "top_domain_under_public_suffix", None) or e.registered_domain or (urlparse(u).hostname or "")
def campaign_key(u):
    path = re.sub(r"[0-9]+", "0", re.sub(r"[a-z0-9]{10,}", "X", urlparse(u).path.lower()))
    return regdom(u) + "|" + path

phish_cl = np.array([campaign_key(u) for u in fresh_phish])
home_cl  = np.array([regdom(u) for u in fresh_home])
paths_cl = np.array([regdom(u) for u in fresh_paths])
print("clusters -> phishing:", len(set(phish_cl)), "| homepage sites:", len(set(home_cl)),
      "| path-URL sites:", len(set(paths_cl)))

hv = HashingVectorizer(analyzer="char", ngram_range=(1, 4), n_features=2**20, alternate_sign=False)
canon = lambda u: re.sub(r"^www\.", "", re.sub(r"^https?://", "", str(u).lower().strip())).rstrip("/")
host = lambda u: re.split(r"[/?#]", canon(u))[0]

def train_text(u, rng, mode):
    if mode == "raw": return u
    c = host(u) if mode == "hostonly" else canon(u)
    return rng.choice(["http://", "https://"]) + rng.choice(["", "www."]) + c

def test_text(u, mode):
    return "https://" + host(u) if mode == "hostonly" else u

tr = legacy.sample(200_000, random_state=0)
ytr = (tr["label"] == "phishing").astype(int)
scores = {}
for mode in ("raw", "augmented", "hostonly"):
    rng = random.Random(0)
    clf = SGDClassifier(loss="log_loss", alpha=1e-6, random_state=0).fit(
        hv.transform([train_text(u, rng, mode) for u in tr["url"]]), ytr)
    P = lambda urls: clf.predict_proba(hv.transform([test_text(u, mode) for u in urls]))[:, 1]
    scores[mode] = {"phish": P(fresh_phish), "home": P(fresh_home), "paths": P(fresh_paths)}
    print("trained", mode)

def groups(cl):
    d = {}
    for i, c in enumerate(cl): d.setdefault(c, []).append(i)
    return [np.array(v) for v in d.values()]

def summarize(pos, neg, pcl, ncl, B=1000, seed=0):
    def metrics(p, n):
        y = np.r_[np.ones(len(p)), np.zeros(len(n))]; s = np.r_[p, n]
        fpr, tpr, _ = roc_curve(y, s)
        return {"TPR@5%FPR": np.interp(0.05, fpr, tpr), "TPR@10%FPR": np.interp(0.10, fpr, tpr),
                "AUC": roc_auc_score(y, s)}
    point = metrics(pos, neg)
    rng = np.random.default_rng(seed); PG, NG = groups(pcl), groups(ncl)
    runs = {k: [] for k in point}
    for _ in range(B):
        pi = np.concatenate([PG[j] for j in rng.integers(0, len(PG), len(PG))])
        ni = np.concatenate([NG[j] for j in rng.integers(0, len(NG), len(NG))])
        for k, v in metrics(pos[pi], neg[ni]).items(): runs[k].append(v)
    return {k: f"{point[k]:.2f} [{np.percentile(runs[k], 2.5):.2f}, {np.percentile(runs[k], 97.5):.2f}]"
            for k in point}

rows = {}
for mode in scores:
    for name, ncl in (("home", home_cl), ("paths", paths_cl)):
        rows[(mode, "vs legit " + name)] = summarize(scores[mode]["phish"], scores[mode][name], phish_cl, ncl)
res = pd.DataFrame(rows).T
print(res.to_string())

res.to_csv("results_cluster_bootstrap.csv")
files.download("results_cluster_bootstrap.csv")

clusters -> phishing: 164 | homepage sites: 300 | path-URL sites: 173
trained raw
trained augmented
trained hostonly
                                  TPR@5%FPR         TPR@10%FPR                AUC
raw       vs legit home   0.00 [0.00, 0.01]  0.01 [0.00, 0.06]  0.52 [0.43, 0.61]
          vs legit paths  0.41 [0.20, 0.56]  0.58 [0.43, 0.70]  0.82 [0.75, 0.87]
augmented vs legit home   0.71 [0.56, 0.81]  0.77 [0.65, 0.87]  0.93 [0.89, 0.95]
          vs legit paths  0.45 [0.29, 0.58]  0.55 [0.42, 0.68]  0.85 [0.79, 0.89]
hostonly  vs legit home   0.58 [0.40, 0.69]  0.61 [0.45, 0.75]  0.86 [0.78, 0.92]
          vs legit paths  0.58 [0.41, 0.70]  0.61 [0.45, 0.75]  0.88 [0.80, 0.93]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
from google.colab import userdata
import subprocess
t = userdata.get("GH_token")
r = subprocess.run(["git", "ls-remote", f"https://{t}@github.com/preranaprakash129-netizen/phishing-url-gnn-detection.git", "HEAD"],
                   capture_output=True, text=True)
print("token works" if r.returncode == 0 else "FAILED: " + r.stderr.replace(t, "***"))

token works


In [1]:
import torch
print("cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

cuda: True | Tesla T4


In [4]:
from google.colab import userdata, files
import os, subprocess, json, glob, zipfile
import pandas as pd

GH_USER, REPO = "preranaprakash129-netizen", "phishing-url-gnn-detection"

if not os.path.exists(REPO):
    token = userdata.get("GH_token")
    r = subprocess.run(["git", "clone", f"https://{token}@github.com/{GH_USER}/{REPO}.git"],
                       capture_output=True, text=True)
    print("clone:", "ok" if r.returncode == 0 else "FAILED")
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://github.com/{GH_USER}/{REPO}.git"])

k = pd.read_csv(f"{REPO}/kaggle_cleaned.csv", usecols=["URL", "label"]).rename(columns={"URL": "url"})
k["label"] = k["label"].map({1: "legitimate", 0: "phishing"})
g1 = pd.read_csv(f"{REPO}/cleaned_github_dataset1.csv")[["url", "label"]]
g2 = pd.DataFrame(json.load(open(f"{REPO}/cleaned_github_dataset_2.json")))[["url", "label"]]
legacy = pd.concat([k, g1, g2], ignore_index=True)
legacy["url"] = legacy["url"].astype(str).str.strip()
legacy = legacy.drop_duplicates(subset="url").reset_index(drop=True)
print("legacy:", legacy.shape)

os.makedirs("fresh", exist_ok=True)
zf = max(glob.glob(f"{REPO}/fresh_backup*.zip"), key=os.path.getsize)
zipfile.ZipFile(zf).extractall("fresh")

recs = [json.loads(l) for p in sorted(glob.glob("fresh/crawl_*.jsonl")) for l in open(p)]
fresh_phish = sorted({r["original_url"] for r in recs if r["klass"] == "phish"})
fresh_home  = sorted({r["original_url"] for r in recs if r["klass"] == "legit"})
fresh_paths = sorted({l for r in recs if r["klass"] == "legit" for l in r["internal_links"]})
print("records:", len(recs), "| phish:", len(fresh_phish), "| homepages:", len(fresh_home),
      "| legit with paths:", len(fresh_paths))

clone: ok
legacy: (743518, 2)
records: 600 | phish: 300 | homepages: 300 | legit with paths: 842


In [5]:
!pip -q install tldextract

import re, random, numpy as np, pandas as pd, torch, torch.nn as nn
from urllib.parse import urlparse
import tldextract
from google.colab import files
from sklearn.metrics import roc_curve, roc_auc_score

assert "legacy" in globals() and "fresh_phish" in globals(), "Run the restore cell first (session was reset)"
dev = "cuda"

# ---------- helpers (same as the linear run) ----------
ext = tldextract.TLDExtract(suffix_list_urls=())
def regdom(u):
    e = ext(u)
    return getattr(e, "top_domain_under_public_suffix", None) or e.registered_domain or (urlparse(u).hostname or "")
def campaign_key(u):
    path = re.sub(r"[0-9]+", "0", re.sub(r"[a-z0-9]{10,}", "X", urlparse(u).path.lower()))
    return regdom(u) + "|" + path

phish_cl = np.array([campaign_key(u) for u in fresh_phish])
home_cl  = np.array([regdom(u) for u in fresh_home])
paths_cl = np.array([regdom(u) for u in fresh_paths])

canon = lambda u: re.sub(r"^www\.", "", re.sub(r"^https?://", "", str(u).lower().strip())).rstrip("/")
host  = lambda u: re.split(r"[/?#]", canon(u))[0]

def train_text(u, rng, mode):
    if mode == "raw": return u
    c = host(u) if mode == "hostonly" else canon(u)
    return rng.choice(["http://", "https://"]) + rng.choice(["", "www."]) + c

def test_text(u, mode):
    return "https://" + host(u) if mode == "hostonly" else u

# ---------- character encoding ----------
L = 200
def enc(texts):
    out = np.zeros((len(texts), L), dtype=np.uint8)
    for i, s in enumerate(texts):
        b = s.lower().encode("ascii", "ignore")[:L]
        out[i, :len(b)] = np.frombuffer(b, dtype=np.uint8)
    return out

# ---------- model ----------
class CharCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(128, 32, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(32, 128, k, padding=k // 2) for k in (3, 5, 7)])
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(384, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x):
        h = self.emb(x).transpose(1, 2)
        h = torch.cat([torch.relu(c(h)).max(dim=2).values for c in self.convs], dim=1)
        return self.head(h).squeeze(1)

@torch.no_grad()
def predict(model, texts):
    model.eval(); X = torch.from_numpy(enc(texts)); out = []
    for i in range(0, len(X), 2048):
        out.append(torch.sigmoid(model(X[i:i + 2048].to(dev).long())).cpu().numpy())
    return np.concatenate(out)

# ---------- data split (same 200k sample as the linear run) ----------
tr = legacy.sample(200_000, random_state=0)
tr_urls = tr["url"].tolist()
ytr = (tr["label"] == "phishing").astype(int).values
held = legacy.drop(tr.index).sample(20_000, random_state=1)
yheld = (held["label"] == "phishing").astype(int).values

def train_cnn(mode, seed=0, epochs=3):
    torch.manual_seed(seed); rng = random.Random(seed)
    model = CharCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    lossf = nn.BCEWithLogitsLoss()
    y = torch.tensor(ytr, dtype=torch.float32)
    for ep in range(epochs):
        X = torch.from_numpy(enc([train_text(u, rng, mode) for u in tr_urls]))
        perm = torch.randperm(len(X)); model.train(); tot = 0.0
        for i in range(0, len(X), 512):
            idx = perm[i:i + 512]
            xb, yb = X[idx].to(dev).long(), y[idx].to(dev)
            opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        print(f"  {mode} epoch {ep + 1}: loss {tot / len(X):.4f}")
    return model

scores = {}
for mode in ("raw", "augmented", "hostonly"):
    print("training", mode)
    m = train_cnn(mode)
    P = lambda urls: predict(m, [test_text(u, mode) for u in urls])
    scores[mode] = {"phish": P(fresh_phish), "home": P(fresh_home), "paths": P(fresh_paths)}
    print(f"  held-out legacy AUC ({mode}): {roc_auc_score(yheld, P(held['url'].tolist())):.3f}")

# ---------- cluster bootstrap (same as the linear run) ----------
def groups(cl):
    d = {}
    for i, c in enumerate(cl): d.setdefault(c, []).append(i)
    return [np.array(v) for v in d.values()]

def summarize(pos, neg, pcl, ncl, B=1000, seed=0):
    def metrics(p, n):
        y = np.r_[np.ones(len(p)), np.zeros(len(n))]; s = np.r_[p, n]
        fpr, tpr, _ = roc_curve(y, s)
        return {"TPR@5%FPR": np.interp(0.05, fpr, tpr), "TPR@10%FPR": np.interp(0.10, fpr, tpr),
                "AUC": roc_auc_score(y, s)}
    point = metrics(pos, neg)
    rng = np.random.default_rng(seed); PG, NG = groups(pcl), groups(ncl)
    runs = {k: [] for k in point}
    for _ in range(B):
        pi = np.concatenate([PG[j] for j in rng.integers(0, len(PG), len(PG))])
        ni = np.concatenate([NG[j] for j in rng.integers(0, len(NG), len(NG))])
        for k, v in metrics(pos[pi], neg[ni]).items(): runs[k].append(v)
    return {k: f"{point[k]:.2f} [{np.percentile(runs[k], 2.5):.2f}, {np.percentile(runs[k], 97.5):.2f}]"
            for k in point}

rows = {}
for mode in scores:
    for name, ncl in (("home", home_cl), ("paths", paths_cl)):
        rows[(mode, "vs legit " + name)] = summarize(scores[mode]["phish"], scores[mode][name], phish_cl, ncl)
res = pd.DataFrame(rows).T
print(res.to_string())

res.to_csv("results_cnn_bootstrap.csv")
files.download("results_cnn_bootstrap.csv")

training raw
  raw epoch 1: loss 0.0674
  raw epoch 2: loss 0.0255
  raw epoch 3: loss 0.0207
  held-out legacy AUC (raw): 0.999
training augmented
  augmented epoch 1: loss 0.3480
  augmented epoch 2: loss 0.2079
  augmented epoch 3: loss 0.1830
  held-out legacy AUC (augmented): 0.978
training hostonly
  hostonly epoch 1: loss 0.4137
  hostonly epoch 2: loss 0.3388
  hostonly epoch 3: loss 0.3189
  held-out legacy AUC (hostonly): 0.929
                                  TPR@5%FPR         TPR@10%FPR                AUC
raw       vs legit home   0.32 [0.21, 0.45]  0.35 [0.24, 0.48]  0.71 [0.62, 0.78]
          vs legit paths  0.53 [0.29, 0.66]  0.64 [0.49, 0.74]  0.84 [0.78, 0.88]
augmented vs legit home   0.73 [0.54, 0.85]  0.88 [0.77, 0.93]  0.95 [0.92, 0.97]
          vs legit paths  0.40 [0.27, 0.54]  0.48 [0.34, 0.61]  0.83 [0.77, 0.87]
hostonly  vs legit home   0.57 [0.41, 0.70]  0.66 [0.49, 0.77]  0.86 [0.78, 0.91]
          vs legit paths  0.59 [0.43, 0.72]  0.65 [0.49, 0.77]  0.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
assert "train_cnn" in globals() and "held" in globals(), "Run the CNN cell first in this session"

def rm_www(u):  return re.sub(r"^(https?://)www\.", r"\1", u)
def to_http(u): return re.sub(r"^https://", "http://", u)
def add_www(u): return re.sub(r"^(https?://)(?!www\.)", r"\1www.", u)

hl = held[held["label"] == "legitimate"]["url"]
hp = held[held["label"] == "phishing"]["url"]
L_www   = hl[hl.str.match(r"^https?://www\.")].sample(min(2000, hl.str.match(r"^https?://www\.").sum()), random_state=0).tolist()
L_https = hl[hl.str.startswith("https://")].sample(min(2000, hl.str.startswith("https://").sum()), random_state=0).tolist()
P_nowww = hp[hp.str.match(r"^https?://(?!www\.)")].sample(min(2000, hp.str.match(r"^https?://(?!www\.)").sum()), random_state=0).tolist()
F_home  = [u for u in fresh_home if re.match(r"^https?://(?!www\.)", u)]
print("perturbation sets:", len(L_www), len(L_https), len(P_nowww), len(F_home))

def point(p, n):
    y = np.r_[np.ones(len(p)), np.zeros(len(n))]; s = np.r_[p, n]
    fpr, tpr, _ = roc_curve(y, s)
    return {"TPR@5%FPR": np.interp(0.05, fpr, tpr), "TPR@10%FPR": np.interp(0.10, fpr, tpr),
            "AUC": roc_auc_score(y, s)}

rate = lambda sc: float((sc > 0.5).mean())
rows = []
for seed in (0, 1, 2):
    for mode in ("raw", "augmented", "hostonly"):
        print(f"seed {seed} | {mode}")
        m = train_cnn(mode, seed=seed)
        P = lambda urls: predict(m, [test_text(u, mode) for u in urls])
        sp, sh, spa = P(fresh_phish), P(fresh_home), P(fresh_paths)
        r = {"seed": seed, "model": mode,
             "held_AUC": roc_auc_score(yheld, P(held["url"].tolist()))}
        for k, v in point(sp, sh).items():  r[f"home_{k}"] = v
        for k, v in point(sp, spa).items(): r[f"paths_{k}"] = v
        r["legit_www_flag_before"]  = rate(P(L_www));   r["legit_www_flag_after"]  = rate(P([rm_www(u) for u in L_www]))
        r["legit_http_flag_before"] = rate(P(L_https)); r["legit_http_flag_after"] = rate(P([to_http(u) for u in L_https]))
        r["phish_www_catch_before"] = rate(P(P_nowww)); r["phish_www_catch_after"] = rate(P([add_www(u) for u in P_nowww]))
        r["fresh_home_flag_before"] = rate(P(F_home));  r["fresh_home_flag_after"] = rate(P([add_www(u) for u in F_home]))
        rows.append(r)

df = pd.DataFrame(rows)
df.to_csv("results_cnn_3seeds_perturb.csv", index=False)
g = df.drop(columns="seed").groupby("model").agg(["mean", "std"]).round(3)
pd.set_option("display.width", 250, "display.max_columns", 50)
print(g.T.to_string())
files.download("results_cnn_3seeds_perturb.csv")

perturbation sets: 2000 2000 2000 300
seed 0 | raw
  raw epoch 1: loss 0.0675
  raw epoch 2: loss 0.0256
  raw epoch 3: loss 0.0208
seed 0 | augmented
  augmented epoch 1: loss 0.3480
  augmented epoch 2: loss 0.2082
  augmented epoch 3: loss 0.1823
seed 0 | hostonly
  hostonly epoch 1: loss 0.4138
  hostonly epoch 2: loss 0.3391
  hostonly epoch 3: loss 0.3182
seed 1 | raw
  raw epoch 1: loss 0.0580
  raw epoch 2: loss 0.0262
  raw epoch 3: loss 0.0205
seed 1 | augmented
  augmented epoch 1: loss 0.3216
  augmented epoch 2: loss 0.2052
  augmented epoch 3: loss 0.1773
seed 1 | hostonly
  hostonly epoch 1: loss 0.4083
  hostonly epoch 2: loss 0.3392
  hostonly epoch 3: loss 0.3152
seed 2 | raw
  raw epoch 1: loss 0.0679
  raw epoch 2: loss 0.0245
  raw epoch 3: loss 0.0203
seed 2 | augmented
  augmented epoch 1: loss 0.3295
  augmented epoch 2: loss 0.2079
  augmented epoch 3: loss 0.1811
seed 2 | hostonly
  hostonly epoch 1: loss 0.4107
  hostonly epoch 2: loss 0.3384
  hostonly epoch

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>